In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
DATA_PATH = Path("AWS_Honeypot_marx-geo.csv")
df = pd.read_csv(DATA_PATH)


In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop("Unnamed: 15", axis=1, inplace=True, errors="ignore")

In [ ]:
def find_day(date):
    return date.strftime("%A")

In [ ]:
def find_month(date):
    return date.strftime("%B")

In [ ]:
def find_year(date):
    return date.year

In [ ]:
def find_hour(date):
    return date.hour

In [ ]:
def find_minute(date):
    return date.minute

In [ ]:
df["datetime"] = df["datetime"].apply(pd.to_datetime)

In [ ]:
df["day"] = df["datetime"].apply(find_day)
df["month"] = df["datetime"].apply(find_month)
df["year"] = df["datetime"].apply(find_year)
df["hour"] = df["datetime"].apply(find_hour)
df["minute"] = df["datetime"].apply(find_minute)

In [ ]:
df.head()

In [ ]:
df["proto"].value_counts().plot(kind="bar")

In [ ]:
df["proto"].value_counts().plot(kind="pie", autopct="%.2f%%")

In [ ]:
df["month"].value_counts().plot(kind="bar").set_title("Cyberattacks per month")

In [ ]:
df["day"].value_counts().plot(kind="pie", autopct="%.2f%%").set_title("Cyberattacks per day")

In [ ]:
df["hour"].value_counts().plot(kind="bar").set_title("Cyberattacks per hour")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, Flatten, Input, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Load Dataset
DATA_PATH = Path("AWS_Honeypot_marx-geo.csv")
df = pd.read_csv(DATA_PATH)
df.drop("Unnamed: 15", axis=1, inplace=True, errors="ignore")  # Drop unused column if present
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")  # Handle parsing errors gracefully
df.dropna(subset=["datetime"], inplace=True)  # Drop rows with invalid datetime

# Feature Engineering
df["day"], df["month"], df["hour"], df["minute"] = (
    df["datetime"].dt.day,
    df["datetime"].dt.month,
    df["datetime"].dt.hour,
    df["datetime"].dt.minute,
)
df["proto"] = LabelEncoder().fit_transform(df["proto"])  # Encode target labels

# Features and Labels
features = df[["spt", "dpt", "hour", "minute", "day", "month"]].fillna(0).values  # Fill missing values
labels = df["proto"].values

# Normalize Features
scaler = MinMaxScaler()
features = scaler.fit_transform(features)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# Reshape for RNN and CNN
X_train_rnn, X_test_rnn = X_train.reshape(X_train.shape[0], 1, X_train.shape[1]), X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
X_train_cnn, X_test_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1), X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Compute Class Weights
class_weights = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)))

# Enhanced RNN Model
model_rnn = Sequential([
    Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])),
    Bidirectional(LSTM(128, activation='relu', return_sequences=True)),
    Dropout(0.4),
    Bidirectional(LSTM(64, activation='relu')),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(len(np.unique(labels)), activation='softmax')
])
model_rnn.compile(optimizer=Adam(learning_rate=0.0005), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_rnn = model_rnn.fit(X_train_rnn, y_train, epochs=30, batch_size=64, validation_split=0.2,
                            class_weight=class_weights, callbacks=[EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)])

# Simplified CNN Model
model_cnn = Sequential([
    Input(shape=(X_train_cnn.shape[1], 1)),
    Conv1D(32, kernel_size=3, activation='relu', padding='same'),  # Reduced filters
    MaxPooling1D(pool_size=2),
    Dropout(0.5),  # Increased Dropout
    Conv1D(64, kernel_size=3, activation='relu', padding='same'),  # Reduced filters
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),  # Reduced Dense layer size
    Dropout(0.5),  # Increased Dropout
    Dense(len(np.unique(labels)), activation='softmax')
])
model_cnn.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_cnn = model_cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.2,
                            class_weight=class_weights, callbacks=[EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)])

# Evaluate Models
rnn_eval = model_rnn.evaluate(X_test_rnn, y_test)
cnn_eval = model_cnn.evaluate(X_test_cnn, y_test)
print(f"RNN Accuracy: {rnn_eval[1] * 100:.2f}%")
print(f"CNN Accuracy: {cnn_eval[1] * 100:.2f}%")

# Plot Training History
def plot_training(history, title):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Plot RNN and CNN histories
plot_training(history_rnn, "RNN")
plot_training(history_cnn, "CNN")


In [ ]:
#  import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import MinMaxScaler, LabelEncoder
# from sklearn.utils.class_weight import compute_class_weight
# import matplotlib.pyplot as plt
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, LSTM, Dropout, Conv1D, MaxPooling1D, Flatten, Input
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping
# from tensorflow.keras.layers import Bidirectional


# # Load Dataset
# df = pd.read_csv("/content/AWS_Honeypot_marx-geo.csv")
# df.drop("Unnamed: 15", axis=1, inplace=True, errors="ignore")  # Drop unused column if present
# df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")  # Handle parsing errors gracefully
# df.dropna(subset=["datetime"], inplace=True)  # Drop rows with invalid datetime

# # Feature Engineering
# df["day"], df["month"], df["hour"], df["minute"] = df["datetime"].dt.day, df["datetime"].dt.month, df["datetime"].dt.hour, df["datetime"].dt.minute
# df["proto"] = LabelEncoder().fit_transform(df["proto"])  # Encode target labels

# # Features and Labels
# features = df[["spt", "dpt", "hour", "minute", "day", "month"]].fillna(0).values  # Fill missing values
# labels = df["proto"].values

# # Normalize Features
# scaler = MinMaxScaler()
# features = scaler.fit_transform(features)

# # Train-Test Split
# X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# # Reshape for RNN and CNN
# X_train_rnn, X_test_rnn = X_train.reshape(X_train.shape[0], 1, X_train.shape[1]), X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
# X_train_cnn, X_test_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1), X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# # Compute Class Weights
# class_weights = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)))

# # RNN Model
# model_rnn = Sequential([
#     Input(shape=(X_train_rnn.shape[1], X_train_rnn.shape[2])),
#     LSTM(128, activation='relu', return_sequences=True),
#     Dropout(0.3),
#     LSTM(64, activation='relu'),
#     Dropout(0.2),
#     Dense(64, activation='relu'),
#     Dense(len(np.unique(labels)), activation='softmax')
# ])
# model_rnn.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history_rnn = model_rnn.fit(X_train_rnn, y_train, epochs=10, batch_size=64, validation_split=0.2,
#                             class_weight=class_weights, callbacks=[EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)])


# # Optimized CNN Model
# model_cnn = Sequential([
#     Input(shape=(X_train_cnn.shape[1], 1)),
#     Conv1D(64, kernel_size=3, activation='relu', padding='same'), # Add padding='same' to the first Conv1D layer
#     MaxPooling1D(pool_size=2),
#     Dropout(0.3),
#     Conv1D(128, kernel_size=3, activation='relu', padding='same'), # Add padding='same' to the second Conv1D layer
#     MaxPooling1D(pool_size=2),
#     Flatten(),
#     Dense(128, activation='relu'),
#     Dropout(0.2),
#     Dense(len(np.unique(labels)), activation='softmax')
# ])

# # Compile the CNN model before evaluating it
# model_cnn.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy']) # This line was missing or commented out
# history_cnn = model_cnn.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.2,
#                             class_weight=class_weights, callbacks=[EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)])

# # ... (rest of the code) ...

# # Evaluate Models
# rnn_eval = model_rnn.evaluate(X_test_rnn, y_test)
# cnn_eval = model_cnn.evaluate(X_test_cnn, y_test)
# print(f"RNN Accuracy: {rnn_eval[1] * 100:.2f}%")
# print(f"CNN Accuracy: {cnn_eval[1] * 100:.2f}%")

# # Plot Training History
# def plot_training(history, title):
#     plt.figure(figsize=(12, 5))
#     plt.subplot(1, 2, 1)
#     plt.plot(history.history['accuracy'], label='Train Acc')
#     plt.plot(history.history['val_accuracy'], label='Val Acc')
#     plt.title(f'{title} - Accuracy')
#     plt.xlabel('Epochs')
#     plt.ylabel('Accuracy')
#     plt.legend()
#     plt.subplot(1, 2, 2)
#     plt.plot(history.history['loss'], label='Train Loss')
#     plt.plot(history.history['val_loss'], label='Val Loss')
#     plt.title(f'{title} - Loss')
#     plt.xlabel('Epochs')
#     plt.ylabel('Loss')
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

# plot_training(history_rnn, "RNN")
# plot_training(history_cnn, "CNN")
